# Lab 15. Forecast Evaluation and Backtesting

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-15-forecast-evaluation-backtesting-lab.ipynb)

This lab is designed for independent study. It gives the background ideas before each programming step and asks you to interpret the output as you go.

In earlier labs, we learned how to build time series models. In this lab, the main question changes:

> How do we decide whether a forecasting method is actually good?

For time series, evaluation is not the same as ordinary random train/test splitting. The order of time matters. A model should only use information that would have been available at the time the forecast was made.

## Learning goals

By the end of this lab, you should be able to:

1. Explain why ordinary random cross-validation is usually inappropriate for forecasting.
2. Define forecast origin, forecast horizon, point forecast error, and rolling-origin backtesting.
3. Implement naive, seasonal naive, moving average, and lag-regression forecasting baselines.
4. Compute MAE, RMSE, MAPE, sMAPE, and MASE.
5. Compare models overall and by forecast horizon.
6. Evaluate simple prediction intervals using empirical coverage.
7. Detect common forms of leakage in time series evaluation.

No command `mathcal` is used in this notebook, so the formulas should display well in Google Colab.

## 1. Mathematical background: forecasts, origins, and horizons

Let $y_1, y_2, \ldots, y_T$ be an observed time series.

A **forecast origin** is a time $t$ at which we pretend the future has not happened yet. A **forecast horizon** $h$ asks us to predict $y_{t+h}$ using information available at time $t$.

We write a forecast as

$$
\widehat y_{t+h \mid t}.
$$

The corresponding forecast error is

$$
e_{t,h} = y_{t+h} - \widehat y_{t+h \mid t}.
$$

The subscript $t+h \mid t$ means: predict time $t+h$ using data available through time $t$.

This notation is important because it helps us avoid leakage. A forecasting method should never use $y_{t+1}, \ldots, y_{t+h}$ when creating $\widehat y_{t+h \mid t}$.

## 2. Setup

We use only common Python packages. The random seed is fixed so your results should be reproducible.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7339)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

## 3. Create a synthetic forecasting dataset

A synthetic dataset is useful because we know the true structure. We will build a monthly series with:

- a slowly increasing trend,
- seasonal behavior with period 12,
- autoregressive dependence in the noise,
- a mild level shift near the end.

This gives us a realistic but controlled forecasting task.

In [ ]:
def simulate_monthly_series(n=180, season_period=12, seed=7339):
    rng = np.random.default_rng(seed)
    t = np.arange(n)

    trend = 50 + 0.18 * t
    seasonal = 7 * np.sin(2 * np.pi * t / season_period) + 2.5 * np.cos(2 * np.pi * t / season_period)

    # AR(1)-type noise
    noise = np.zeros(n)
    innovations = rng.normal(0, 2.0, size=n)
    for i in range(1, n):
        noise[i] = 0.55 * noise[i - 1] + innovations[i]

    # A mild level shift after month 125
    shift = np.where(t >= 125, 6.0, 0.0)

    y = trend + seasonal + noise + shift
    dates = pd.date_range("2010-01-01", periods=n, freq="MS")
    return pd.Series(y, index=dates, name="y")

series = simulate_monthly_series()
series.head(), series.tail()

In [ ]:
ax = series.plot(title="Synthetic monthly time series")
ax.set_xlabel("date")
ax.set_ylabel("value")
plt.show()

### Checkpoint 1

Before fitting any model, answer these questions:

1. Does the series appear stationary?
2. Is there a seasonal pattern?
3. Would a random train/test split be reasonable here? Why or why not?

## 4. Chronological train/test split

A basic forecasting split uses the earlier part of the series for training and the later part for testing.

Here we reserve the last 36 months as the test set. This corresponds to evaluating how well models forecast the final three years.

In [ ]:
test_size = 36
train = series.iloc[:-test_size]
test = series.iloc[-test_size:]

print("Training range:", train.index.min().date(), "to", train.index.max().date(), "n =", len(train))
print("Test range:    ", test.index.min().date(), "to", test.index.max().date(), "n =", len(test))

ax = train.plot(label="train")
test.plot(ax=ax, label="test")
ax.axvline(test.index[0], linestyle="--")
ax.set_title("Chronological train/test split")
ax.legend()
plt.show()

## 5. Baseline forecasting methods

Good forecasting projects always start with simple baselines.

We will implement four baselines:

1. **Mean forecast:** predict the average of the training set.
2. **Naive forecast:** predict the last observed value.
3. **Seasonal naive forecast:** predict the value from the same season in the previous cycle.
4. **Moving-average forecast:** predict the average of the most recent window.

A complicated model should be considered useful only if it beats reasonable baselines under honest evaluation.

In [ ]:
def mean_forecast(history, horizon):
    return np.repeat(np.mean(history), horizon)


def naive_forecast(history, horizon):
    return np.repeat(history[-1], horizon)


def seasonal_naive_forecast(history, horizon, season_period=12):
    history = np.asarray(history)
    preds = []
    for h in range(1, horizon + 1):
        idx = -season_period + ((h - 1) % season_period)
        preds.append(history[idx])
    return np.array(preds)


def moving_average_forecast(history, horizon, window=12):
    return np.repeat(np.mean(history[-window:]), horizon)


horizon = len(test)
forecast_df = pd.DataFrame(index=test.index)
forecast_df["actual"] = test.values
forecast_df["mean"] = mean_forecast(train.values, horizon)
forecast_df["naive"] = naive_forecast(train.values, horizon)
forecast_df["seasonal_naive"] = seasonal_naive_forecast(train.values, horizon, season_period=12)
forecast_df["moving_average_12"] = moving_average_forecast(train.values, horizon, window=12)
forecast_df.head()

In [ ]:
ax = train.iloc[-60:].plot(label="recent train")
forecast_df["actual"].plot(ax=ax, label="actual test")
for col in ["mean", "naive", "seasonal_naive", "moving_average_12"]:
    forecast_df[col].plot(ax=ax, label=col)
ax.set_title("Simple baseline forecasts on the test period")
ax.legend()
plt.show()

## 6. Forecast accuracy metrics

For point forecasts, common metrics include:

**Mean absolute error**

$$
\text{MAE} = \frac{1}{n}\sum_{i=1}^n |y_i - \widehat y_i|.
$$

**Root mean squared error**

$$
\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^n (y_i - \widehat y_i)^2}.
$$

**Mean absolute percentage error**

$$
\text{MAPE} = \frac{100}{n}\sum_{i=1}^n \left|\frac{y_i - \widehat y_i}{y_i}\right|.
$$

MAPE is easy to explain but can be unstable when true values are close to zero.

**Symmetric MAPE**

$$
\text{sMAPE} = \frac{100}{n}\sum_{i=1}^n \frac{2|y_i - \widehat y_i|}{|y_i| + |\widehat y_i|}.
$$

**Mean absolute scaled error** compares a model to a naive in-sample benchmark:

$$
\text{MASE} = \frac{\text{mean absolute forecast error}}{\text{mean absolute in-sample naive error}}.
$$

For seasonal data, the denominator often uses seasonal naive errors.

In [ ]:
def mae(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(np.abs(y_true - y_pred))


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = np.abs(y_true) > 1e-12
    return 100 * np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))


def smape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom > 1e-12
    return 100 * np.mean(2 * np.abs(y_true[mask] - y_pred[mask]) / denom[mask])


def mase(y_true, y_pred, train_values, season_period=1):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    train_values = np.asarray(train_values)
    denom = np.mean(np.abs(train_values[season_period:] - train_values[:-season_period]))
    return np.mean(np.abs(y_true - y_pred)) / denom


def evaluate_forecasts(df, actual_col="actual", train_values=None, season_period=12):
    rows = []
    actual = df[actual_col].values
    for col in df.columns:
        if col == actual_col:
            continue
        pred = df[col].values
        rows.append({
            "model": col,
            "MAE": mae(actual, pred),
            "RMSE": rmse(actual, pred),
            "MAPE": mape(actual, pred),
            "sMAPE": smape(actual, pred),
            "MASE seasonal": mase(actual, pred, train_values, season_period=season_period)
        })
    return pd.DataFrame(rows).sort_values("MAE")

metrics_table = evaluate_forecasts(forecast_df, train_values=train.values, season_period=12)
metrics_table

### Checkpoint 2

Interpret the metrics table.

1. Which baseline has the smallest MAE?
2. Which baseline has the smallest RMSE?
3. Do the metrics agree?
4. Why might a seasonal baseline be hard to beat for a strongly seasonal series?

## 7. Rolling-origin backtesting

A single train/test split is useful, but it gives only one forecast experiment.

In rolling-origin evaluation, we repeat the forecasting experiment at many forecast origins. For each origin $t$:

1. Fit or update the model using data up to $t$.
2. Forecast horizons $1,2,\ldots,H$.
3. Store the errors.
4. Move the forecast origin forward.

This gives a more reliable estimate of forecasting performance.

In [ ]:
def lag_regression_forecast(history, horizon, lags=12, ridge=1e-6):
    '''Recursive autoregressive forecasting using least squares on lag features.'''
    history = np.asarray(history, dtype=float)

    if len(history) <= lags + 5:
        return naive_forecast(history, horizon)

    X = []
    y = []
    for t in range(lags, len(history)):
        X.append(history[t - lags:t][::-1])
        y.append(history[t])
    X = np.asarray(X)
    y = np.asarray(y)

    # Add intercept and use a tiny ridge penalty for numerical stability.
    X_design = np.column_stack([np.ones(len(X)), X])
    penalty = ridge * np.eye(X_design.shape[1])
    penalty[0, 0] = 0.0
    beta = np.linalg.solve(X_design.T @ X_design + penalty, X_design.T @ y)

    preds = []
    values = list(history)
    for _ in range(horizon):
        features = np.array(values[-lags:][::-1])
        x_new = np.r_[1.0, features]
        pred = float(x_new @ beta)
        preds.append(pred)
        values.append(pred)

    return np.array(preds)


def make_forecast(history, model_name, horizon, season_period=12):
    if model_name == "naive":
        return naive_forecast(history, horizon)
    if model_name == "seasonal_naive":
        return seasonal_naive_forecast(history, horizon, season_period=season_period)
    if model_name == "moving_average_12":
        return moving_average_forecast(history, horizon, window=12)
    if model_name == "lag_regression_12":
        return lag_regression_forecast(history, horizon, lags=12)
    raise ValueError(f"Unknown model: {model_name}")


def rolling_origin_backtest(y, models, initial_train_size, horizon, step=1, season_period=12):
    y = pd.Series(y).copy()
    records = []

    max_origin = len(y) - horizon
    for origin in range(initial_train_size, max_origin + 1, step):
        history = y.iloc[:origin].values
        future = y.iloc[origin:origin + horizon].values
        future_index = y.index[origin:origin + horizon]

        for model_name in models:
            preds = make_forecast(history, model_name, horizon, season_period=season_period)
            for h in range(1, horizon + 1):
                records.append({
                    "origin_index": origin,
                    "origin_date": y.index[origin - 1],
                    "target_date": future_index[h - 1],
                    "horizon": h,
                    "model": model_name,
                    "actual": future[h - 1],
                    "forecast": preds[h - 1],
                    "error": future[h - 1] - preds[h - 1]
                })

    return pd.DataFrame(records)

models = ["naive", "seasonal_naive", "moving_average_12", "lag_regression_12"]
backtest = rolling_origin_backtest(
    series,
    models=models,
    initial_train_size=84,
    horizon=12,
    step=3,
    season_period=12
)

backtest.head(), backtest.shape

In [ ]:
def summarize_backtest(bt, train_values, season_period=12):
    rows = []
    for model_name, group in bt.groupby("model"):
        rows.append({
            "model": model_name,
            "MAE": mae(group["actual"], group["forecast"]),
            "RMSE": rmse(group["actual"], group["forecast"]),
            "MAPE": mape(group["actual"], group["forecast"]),
            "sMAPE": smape(group["actual"], group["forecast"]),
            "MASE seasonal": mase(group["actual"], group["forecast"], train_values, season_period=season_period)
        })
    return pd.DataFrame(rows).sort_values("MAE")

bt_summary = summarize_backtest(backtest, train.values, season_period=12)
bt_summary

In [ ]:
ax = bt_summary.set_index("model")[["MAE", "RMSE"]].plot(kind="bar")
ax.set_title("Rolling-origin backtest: MAE and RMSE by model")
ax.set_ylabel("error")
plt.xticks(rotation=30, ha="right")
plt.show()

### Checkpoint 3

Compare the single train/test result with the rolling-origin result.

1. Did the best model change?
2. Why might rolling-origin evaluation be more reliable than a single split?
3. What are the tradeoffs? Think about computation time and implementation complexity.

## 8. Horizon-wise evaluation

A model may be excellent for one-step-ahead prediction but poor for twelve-step-ahead prediction.

For this reason, we should summarize performance by forecast horizon.

In [ ]:
horizon_rows = []
for (model_name, h), group in backtest.groupby(["model", "horizon"]):
    horizon_rows.append({
        "model": model_name,
        "horizon": h,
        "MAE": mae(group["actual"], group["forecast"]),
        "RMSE": rmse(group["actual"], group["forecast"])
    })

horizon_summary = pd.DataFrame(horizon_rows)
horizon_summary.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for model_name, group in horizon_summary.groupby("model"):
    ax.plot(group["horizon"], group["MAE"], marker="o", label=model_name)
ax.set_title("MAE by forecast horizon")
ax.set_xlabel("forecast horizon")
ax.set_ylabel("MAE")
ax.legend()
plt.show()

### Checkpoint 4

Which model performs best at short horizons? Which model performs best at long horizons?

It is common for different models to win at different horizons. This is one reason production forecasting systems often store horizon-wise metrics rather than only one aggregate number.

## 9. A leakage warning: centered rolling features

A common mistake is to use future data while constructing features.

For example, a centered rolling mean at time $t$ may use values from both before and after $t$. That is not available at forecast time. Such features can make validation performance look unrealistically good.

The following demonstration compares:

- a safe trailing rolling mean, which uses only current and past values;
- an unsafe centered rolling mean, which uses future values.

In [ ]:
leak_df = pd.DataFrame({"y": series})
leak_df["safe_trailing_mean"] = leak_df["y"].rolling(window=12, min_periods=12).mean()
leak_df["unsafe_centered_mean"] = leak_df["y"].rolling(window=12, center=True, min_periods=12).mean()

# We compare how closely these smoothed values track the original series after dropping missing values.
safe_part = leak_df.dropna(subset=["safe_trailing_mean"])
unsafe_part = leak_df.dropna(subset=["unsafe_centered_mean"])

safe_error = mae(safe_part["y"], safe_part["safe_trailing_mean"])
unsafe_error = mae(unsafe_part["y"], unsafe_part["unsafe_centered_mean"])

print("MAE of safe trailing rolling mean against current y:  ", round(safe_error, 3))
print("MAE of unsafe centered rolling mean against current y:", round(unsafe_error, 3))

ax = leak_df[["y", "safe_trailing_mean", "unsafe_centered_mean"]].iloc[60:110].plot()
ax.set_title("Safe trailing mean versus unsafe centered mean")
plt.show()

The centered rolling mean often looks better because it can use future observations. That is useful for retrospective smoothing, but it is not valid for forecasting evaluation.

## 10. Prediction intervals and empirical coverage

Point forecasts are not enough. A forecast should often communicate uncertainty.

A simple empirical interval method is:

1. Choose a forecasting method, such as seasonal naive.
2. Compute past forecast residuals from rolling-origin backtesting.
3. Use residual quantiles to form intervals around future point forecasts.

For an approximate 90% interval, use the 5% and 95% residual quantiles:

$$
[\widehat y + q_{0.05}, \widehat y + q_{0.95}].
$$

This is not a perfect probabilistic model, but it is a practical starting point.

In [ ]:
def interval_coverage(actual, lower, upper):
    actual = np.asarray(actual)
    lower = np.asarray(lower)
    upper = np.asarray(upper)
    return np.mean((actual >= lower) & (actual <= upper))

# Use residuals from seasonal naive backtest to create horizon-specific intervals.
sn_bt = backtest[backtest["model"] == "seasonal_naive"].copy()
interval_rows = []
for h, group in sn_bt.groupby("horizon"):
    q05, q95 = np.quantile(group["error"], [0.05, 0.95])
    lower = group["forecast"] + q05
    upper = group["forecast"] + q95
    interval_rows.append({
        "horizon": h,
        "q05 residual": q05,
        "q95 residual": q95,
        "coverage": interval_coverage(group["actual"], lower, upper),
        "average width": np.mean(upper - lower)
    })

interval_summary = pd.DataFrame(interval_rows)
interval_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(interval_summary["horizon"], interval_summary["coverage"], marker="o")
ax.axhline(0.90, linestyle="--", label="nominal 90%")
ax.set_ylim(0, 1.05)
ax.set_title("Empirical interval coverage by horizon")
ax.set_xlabel("forecast horizon")
ax.set_ylabel("coverage")
ax.legend()
plt.show()

### Checkpoint 5

Does the empirical coverage stay close to 90%? If not, what might explain the difference?

Possible reasons include:

- the synthetic series has a level shift;
- the residual distribution changes over time;
- residuals may be dependent;
- the interval method is simple and does not model changing uncertainty.

## 11. Pinball loss for quantile forecasts

For probabilistic forecasting, quantile forecasts are often evaluated using **pinball loss**.

For quantile level $\alpha$, true value $y$, and quantile forecast $q$, the loss is

$$
L_\alpha(y,q) =
\begin{cases}
\alpha(y-q), & y \ge q, \\
(1-\alpha)(q-y), & y < q.
\end{cases}
$$

Lower pinball loss is better.

In [ ]:
def pinball_loss(y_true, q_pred, alpha):
    y_true = np.asarray(y_true)
    q_pred = np.asarray(q_pred)
    diff = y_true - q_pred
    return np.mean(np.maximum(alpha * diff, (alpha - 1) * diff))

# Example: evaluate the lower and upper empirical interval endpoints as quantile forecasts.
pinball_rows = []
for h, group in sn_bt.groupby("horizon"):
    q05, q95 = np.quantile(group["error"], [0.05, 0.95])
    lower_q = group["forecast"] + q05
    upper_q = group["forecast"] + q95
    pinball_rows.append({
        "horizon": h,
        "pinball 0.05": pinball_loss(group["actual"], lower_q, 0.05),
        "pinball 0.95": pinball_loss(group["actual"], upper_q, 0.95)
    })

pd.DataFrame(pinball_rows).head()

## 12. Practice: design your own backtest

Modify the code above to answer at least two of the following questions:

1. What happens if the forecast horizon is 6 instead of 12?
2. What happens if the rolling-origin step is 1 instead of 3?
3. What happens if the initial training size is 120 instead of 84?
4. What happens if lag regression uses 24 lags instead of 12?
5. Which model is best according to MASE rather than MAE?

Write two or three sentences explaining your findings.

In [ ]:
# Student workspace.
# Try changing horizon, step, initial_train_size, or model settings.

practice_bt = rolling_origin_backtest(
    series,
    models=models,
    initial_train_size=96,
    horizon=6,
    step=3,
    season_period=12
)

summarize_backtest(practice_bt, train.values, season_period=12)

## 13. AI-assisted study prompts

Use an AI assistant as a reviewer, not as an authority. Paste your code and results, then ask questions like these:

1. "Check whether my rolling-origin backtest uses future data anywhere. Be specific about possible leakage points."
2. "Explain why my seasonal naive model beats my lag-regression model on this dataset. Give three possible reasons."
3. "I computed MAE, RMSE, sMAPE, and MASE. Explain what each metric emphasizes and when it can be misleading."
4. "Review my prediction interval code. Does it estimate empirical coverage correctly?"
5. "Suggest a better baseline for a monthly seasonal series with trend, but do not use future test values."

After receiving an AI answer, verify the claims by checking the code, rerunning cells, or creating a smaller example where you know the correct answer.

## 14. Mini-project

Choose one of the following tasks.

### Option A. Forecasting benchmark report

Use the synthetic series from this lab. Compare at least four models using rolling-origin backtesting. Report:

- overall MAE, RMSE, and MASE;
- horizon-wise MAE;
- one plot of forecasts from a selected origin;
- a short discussion of the best baseline.

### Option B. Leakage audit

Create a deliberately flawed forecasting feature that uses future information. Compare it with a leakage-safe version. Explain why the flawed version gives an unfair estimate of performance.

### Option C. Interval evaluation

Construct 80%, 90%, and 95% empirical residual intervals for one model. Compare empirical coverage and average interval width.

## 15. Lab checklist

Before leaving this lab, make sure you can do the following without looking at the solution:

- Define forecast origin and forecast horizon.
- Explain why random train/test splitting is usually inappropriate for forecasting.
- Implement naive and seasonal naive forecasts.
- Compute MAE, RMSE, sMAPE, and MASE.
- Run a rolling-origin backtest.
- Summarize errors by horizon.
- Explain one common leakage mistake.
- Evaluate empirical interval coverage.